In [ ]:
import pandas
import numpy
import os

In [ ]:
Amphibian = pandas.read_csv("sound_dataset/Amphibian_Sounds.csv")

In [ ]:
Amphibian

In [ ]:
for i in Amphibian.loc:
    print(i["id"]+"/"+i["taxon_kingdom_name"]+"/"+i["taxon_class_name"]+"/"+i["taxon_order_name"])
    break

In [ ]:
Amphibian.loc

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd
import requests
from tqdm import tqdm


def download_and_rename_inat_dataset(
    csv_path, url_col, name_cols, output_folder="inaturalist_sounds"
):
    """Downloads audio files from an iNaturalist dataset and renames them dynamically.

    Parameters:
    - csv_path (str): Path to your CSV dataset file.
    - url_col (str): The column name containing the static.inaturalist.org
    links.
    - name_cols (list): A list of column names to build your custom filename
      (e.g., ['id', 'taxon_species_name', 'user_login']).
    - output_folder (str): Target folder to save the downloaded audio.
    """
    # 1. Create target folder
    out_dir = Path(output_folder)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 2. Read the dataset
    print(f"Loading dataset from {csv_path}...")
    df = csv_path

    # Clean out any rows missing a URL
    df = df.dropna(subset=[url_col])

    print(f"Found {len(df)} audio records. Starting downloads...\n")

    # 3. Iterate through rows with a progress bar
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Downloading"):
        url = str(row[url_col]).strip()

        try:
            # 4. Extract the native extension from URL safely (handles query params)
            parsed_url = urlparse(url)
            clean_path = Path(parsed_url.path)
            extension = clean_path.suffix

            if not extension:
                extension = ".m4a"  # Fallback default for iNaturalist

            # 5. Construct your preferred custom filename from specified columns
            # Sanitize values to replace characters that aren't allowed in OS filenames
            name_parts = []
            for col in name_cols:
                val = str(row[col]).strip().replace("/", "-").replace("\\", "-")
                name_parts.append(val)

            # Stitch parts together with underscores (e.g., "229366_Passer_domesticus_johndoe.m4a")
            custom_filename = "_".join(name_parts) + extension
            destination_path = out_dir / custom_filename

            # Skip downloading if the file already exists (great for resuming interrupted tasks)
            if destination_path.exists():
                continue

            # 6. Stream file content down to disk
            response = requests.get(url, stream=True, timeout=15)
            if response.status_code == 200:
                with open(destination_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
            else:
                print(
                    f"\n[Warning] Failed to download row {index}: Status code {response.status_code}"
                )

        except Exception as e:
            print(f"\n[Error] Could not process row {index} due to: {e}")

In [ ]:
download_and_rename_inat_dataset(csv_path=Amphibian,url_col="sound_url",name_cols=["id", "taxon_kingdom_name", "taxon_class_name", "taxon_order_name"],output_folder= "Amphibian_sounds")

In [ ]:
import os

# Define the path to your dataset folder
folder_path = "Amphibian_sounds"

# Create an empty list to store the file names
file_names = []

# Loop through all files in the directory
for filename in os.listdir(folder_path):
    # Optional: ensure we only grab files (not subdirectories)
    if os.path.isfile(os.path.join(folder_path, filename)):
        file_names.append(filename)

#print(file_names)
Filenames = pandas.DataFrame(file_names)

In [ ]:
Filenames

In [ ]:
Filenames["filepath"] = Filenames[0].apply(lambda x: x.split("_")[0])

In [ ]:
Filenames

In [ ]:
Amphibians = Amphibian.set_index("id")

In [ ]:
Filenames = Filenames.set_index("filepath")

In [ ]:
Amphibian.merge(left_index = True)

In [ ]:
Amphibians

In [ ]:
Filenames = Filenames.reset_index().rename(columns = {"filepath" : "id"}).set_index("id")

In [ ]:
Filenames

In [ ]:
pandas.merge(Amphibians, Filenames,left_index = True, right_index = True)

In [ ]:
merged_df = pandas.merge(
    Amphibians, Filenames, left_index=True, right_index=True, how="inn"
)

In [ ]:
merged_df

In [126]:
Filenames.index.dtype

dtype('O')

In [127]:
Amphibians.index = Amphibians.index.astype(str)

In [128]:
Amphibians.index.dtype

dtype('O')

In [129]:
merged_df = pandas.merge(
    Amphibians, Filenames, left_index=True, right_index=True, how="inner"
)

In [142]:
merged_df = merged_df.rename(columns = {0: "filepath"})

In [143]:
merged_df

,uuid,observed_on,time_observed_at,user_id,quality_grade,url,image_url,sound_url,num_identification_agreements,num_identification_disagreements,...,species_guess,scientific_name,common_name,iconic_taxon_name,taxon_id,taxon_kingdom_name,taxon_phylum_name,taxon_class_name,taxon_order_name,filepath
id,,,,,,,,,,,,,,,,,,,,,
257511044,cd2b8425-a2a9-418b-8484-9c7fcc5ee363,2025-01-03,2025-01-03 20:33:00 UTC,9531,research,https://www.inaturalist.org/observations/25751...,NaN,https://static.inaturalist.org/sounds/1280478....,1,0,...,Southern Leopard Frog,Lithobates sphenocephalus,Southern Leopard Frog,Amphibia,60341,Animalia,Chordata,Amphibia,Anura,257511044_Animalia_Amphibia_Anura.wav
257512800,948ae9fb-5451-4dc6-a84a-dc892cec3406,2025-01-04,2025-01-04 20:58:00 UTC,100933,research,https://www.inaturalist.org/observations/25751...,NaN,https://static.inaturalist.org/sounds/1280490....,2,0,...,Pacific chorus frog,Pseudacris regilla,Pacific chorus frog,Amphibia,24259,Animalia,Chordata,Amphibia,Anura,257512800_Animalia_Amphibia_Anura.m4a
257513151,3e06c819-e484-4b20-ba8d-3d7aa94695c1,2025-01-04,2025-01-05 01:21:00 UTC,100933,research,https://www.inaturalist.org/observations/25751...,NaN,https://static.inaturalist.org/sounds/1280490....,2,0,...,Pacific chorus frog,Pseudacris regilla,Pacific chorus frog,Amphibia,24259,Animalia,Chordata,Amphibia,Anura,257513151_Animalia_Amphibia_Anura.m4a
257637660,b9f5d5d7-6ab8-463f-a28d-28adb5e4115a,2025-01-06,2025-01-06 21:52:02 UTC,1547851,research,https://www.inaturalist.org/observations/25763...,NaN,https://static.inaturalist.org/sounds/1284184....,2,0,...,Pacific chorus frog,Pseudacris regilla,Pacific chorus frog,Amphibia,24259,Animalia,Chordata,Amphibia,Anura,257637660_Animalia_Amphibia_Anura.m4a
257945656,4d202290-1892-44fa-a386-10c35d643c05,2025-01-09,2025-01-09 19:25:23 UTC,5597546,research,https://www.inaturalist.org/observations/25794...,NaN,https://static.inaturalist.org/sounds/1282838....,2,0,...,Pacific chorus frog,Pseudacris regilla,Pacific chorus frog,Amphibia,24259,Animalia,Chordata,Amphibia,Anura,257945656_Animalia_Amphibia_Anura.m4a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
364459588,3a0e9a28-6a83-40c8-a5da-d9a006116266,2025-02-08,2025-02-08 16:00:00 UTC,1064292,research,https://www.inaturalist.org/observations/36445...,https://inaturalist-open-data.s3.amazonaws.com...,https://static.inaturalist.org/sounds/1973688....,1,0,...,Wood Frog,Lithobates sylvaticus,Wood Frog,Amphibia,66012,Animalia,Chordata,Amphibia,Anura,364459588_Animalia_Amphibia_Anura.m4a
364459589,7321657a-2df4-44e9-aa11-b795295a6906,2025-03-31,2025-03-31 10:00:00 UTC,1064292,research,https://www.inaturalist.org/observations/36445...,https://inaturalist-open-data.s3.amazonaws.com...,https://static.inaturalist.org/sounds/1973686....,1,0,...,Southern Leopard Frog,Lithobates sphenocephalus,Southern Leopard Frog,Amphibia,60341,Animalia,Chordata,Amphibia,Anura,364459589_Animalia_Amphibia_Anura.m4a
364459591,a29f0cf3-d7af-4f32-b4a4-ac2d399ee70d,2025-06-14,2025-06-14 07:00:00 UTC,1064292,research,https://www.inaturalist.org/observations/36445...,https://inaturalist-open-data.s3.amazonaws.com...,https://static.inaturalist.org/sounds/1973687....,1,0,...,Cope's Gray Tree Frog,Dryophytes chrysoscelis,Cope's Gray Tree Frog,Amphibia,1668922,Animalia,Chordata,Amphibia,Anura,364459591_Animalia_Amphibia_Anura.m4a


In [125]:
Filenames.loc["257511044"]

0    257511044_Animalia_Amphibia_Anura.wav
Name: 257511044, dtype: object

In [146]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4699 entries, 257511044 to 365056559
Data columns (total 22 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   uuid                              4699 non-null   object 
 1   observed_on                       4699 non-null   object 
 2   time_observed_at                  4534 non-null   object 
 3   user_id                           4699 non-null   int64  
 4   quality_grade                     4699 non-null   object 
 5   url                               4699 non-null   object 
 6   image_url                         271 non-null    object 
 7   sound_url                         4699 non-null   object 
 8   num_identification_agreements     4699 non-null   int64  
 9   num_identification_disagreements  4699 non-null   int64  
 10  place_guess                       4699 non-null   object 
 11  private_place_guess               0 non-null      float64
 12

In [150]:
merged_df["taxon_class_name"].unique()

array(['Amphibia'], dtype=object)

In [152]:
"""
add_family_labels.py

Adds 'family' and 'family_common' columns to the amphibian merged_df.
Extracts genus from scientific_name, maps to family using a comprehensive lookup.

Run this BEFORE the classifier:
    python add_family_labels.py

It reads merged_df.csv and writes merged_df_with_family.csv.
"""

import pandas as pd
import numpy as np

# =============================================================================
# GENUS → FAMILY MAPPING  (covers iNaturalist's most-recorded amphibians)
# =============================================================================

GENUS_TO_FAMILY = {
    # ===== ANURA (Frogs & Toads) =====

    # Ranidae — True Frogs
    **{g: "Ranidae" for g in [
        "Rana", "Lithobates", "Glandirana", "Pelophylax", "Hylarana",
        "Amietia", "Amnirana", "Babina", "Clinotarsus", "Meristogenys",
        "Odorrana", "Papurana", "Ptychadena", "Sanguirana", "Staurois",
        "Sylvirana", "Abavorana", "Sumaterana",
    ]},

    # Hylidae — Tree Frogs
    **{g: "Hylidae" for g in [
        "Hyla", "Dryophytes", "Pseudacris", "Acris", "Smilisca",
        "Agalychnis", "Dendropsophus", "Boana", "Scinax", "Trachycephalus",
        "Osteopilus", "Isthmohyla", "Charadrahyla", "Ecnomiohyla",
        "Exerodonta", "Plectrohyla", "Ptychohyla", "Rheohyla",
        "Triprion", "Anotheca", "Bromeliohyla", "Duellmanohyla",
        "Megastomatohyla", "Quilticohyla", "Sarsinohyla", "Atlantihyla",
        "Tlalocohyla", "Diaglena", "Hyliola",
    ]},

    # Bufonidae — True Toads
    **{g: "Bufonidae" for g in [
        "Anaxyrus", "Bufo", "Rhinella", "Incilius", "Sclerophrys",
        "Duttaphrynus", "Bufotes", "Epidalea", "Atelopus",
        "Amazophrynella", "Rhaebo", "Phrynoidis", "Ansonia",
        "Capensibufo", "Schismaderma", "Nannophryne",
    ]},

    # Eleutherodactylidae — Rain Frogs / Coquís
    **{g: "Eleutherodactylidae" for g in [
        "Eleutherodactylus", "Diasporus", "Syrrhophus",
    ]},

    # Craugastoridae — Fleshbelly Frogs
    **{g: "Craugastoridae" for g in [
        "Craugastor", "Haddadus", "Pristimantis", "Strabomantis",
        "Yunganastes", "Noblella", "Oreobates", "Psychrophrynella",
    ]},

    # Microhylidae — Narrow-mouthed Frogs
    **{g: "Microhylidae" for g in [
        "Gastrophryne", "Hypopachus", "Microhyla", "Kaloula",
        "Chiasmocleis", "Dermatonotus", "Elachistocleis", "Hamptophryne",
        "Kalophrynus", "Micryletta", "Ramanella", "Uperodon",
        "Breviceps", "Cophixalus", "Oreophryne",
    ]},

    # Leptodactylidae — Thin-toed Frogs
    **{g: "Leptodactylidae" for g in [
        "Leptodactylus", "Physalaemus", "Engystomops", "Pleurodema",
        "Adenomera", "Lithodytes",
    ]},

    # Dendrobatidae — Poison Dart Frogs
    **{g: "Dendrobatidae" for g in [
        "Dendrobates", "Oophaga", "Ranitomeya", "Epipedobates",
        "Ameerega", "Phyllobates", "Andinobates", "Colostethus",
        "Silverstoneia", "Hyloxalus",
    ]},

    # Scaphiopodidae — North American Spadefoot Toads
    **{g: "Scaphiopodidae" for g in [
        "Scaphiopus", "Spea",
    ]},

    # Pipidae — Tongueless Frogs
    **{g: "Pipidae" for g in [
        "Xenopus", "Pipa", "Hymenochirus", "Silurana",
    ]},

    # Bombinatoridae — Fire-bellied Toads
    **{g: "Bombinatoridae" for g in [
        "Bombina",
    ]},

    # Alytidae — Midwife Toads & Painted Frogs
    **{g: "Alytidae" for g in [
        "Alytes", "Discoglossus",
    ]},

    # Centrolenidae — Glass Frogs
    **{g: "Centrolenidae" for g in [
        "Centrolene", "Hyalinobatrachium", "Espadarana", "Nymphargus",
        "Sachatamia", "Teratohyla", "Cochranella",
    ]},

    # Rhacophoridae — Old World Tree Frogs / Shrub Frogs
    **{g: "Rhacophoridae" for g in [
        "Rhacophorus", "Polypedates", "Kurixalus", "Zhangixalus",
        "Chiromantis", "Feihyla", "Taruga", "Theloderma",
    ]},

    # Myobatrachidae — Australian Ground Frogs
    **{g: "Myobatrachidae" for g in [
        "Crinia", "Geocrinia", "Myobatrachus", "Pseudophryne",
        "Paracrinia", "Uperoleia",
    ]},

    # Pelodryadidae — Australasian Tree Frogs
    **{g: "Pelodryadidae" for g in [
        "Litoria", "Ranoidea", "Nyctimystes",
    ]},

    # Limnodynastidae
    **{g: "Limnodynastidae" for g in [
        "Limnodynastes", "Platyplectrum", "Adelotus", "Heleioporus",
        "Neobatrachus",
    ]},

    # Pelodytidae — Parsley Frogs
    **{g: "Pelodytidae" for g in [
        "Pelodytes",
    ]},

    # Pelobatidae — European Spadefoot Toads
    **{g: "Pelobatidae" for g in [
        "Pelobates",
    ]},

    # Arthroleptidae — Squeakers
    **{g: "Arthroleptidae" for g in [
        "Arthroleptis", "Leptopelis",
    ]},

    # Hyperoliidae — African Reed Frogs
    **{g: "Hyperoliidae" for g in [
        "Hyperolius", "Afrixalus", "Heterixalus", "Kassina", "Semnodactylus",
    ]},

    # Mantellidae — Malagasy Frogs
    **{g: "Mantellidae" for g in [
        "Mantella", "Boophis", "Mantidactylus", "Guibemantis", "Spinomantis",
    ]},

    # Hemisotidae — Shovelnose Frogs
    **{g: "Hemisotidae" for g in [
        "Hemisus",
    ]},

    # Ceratophryidae — Horned Frogs
    **{g: "Ceratophryidae" for g in [
        "Ceratophrys", "Chacophrys", "Lepidobatrachus",
    ]},

    # Dicroglossidae — Fork-tongued Frogs
    **{g: "Dicroglossidae" for g in [
        "Fejervarya", "Limnonectes", "Hoplobatrachus", "Euphlyctis",
        "Minervarya", "Occidozyga",
    ]},

    # Pyxicephalidae — African Bullfrogs
    **{g: "Pyxicephalidae" for g in [
        "Pyxicephalus", "Tomopterna", "Strongylopus", "Cacosternum",
        "Natalobatrachus",
    ]},

    # Phyllomedusidae — Leaf Frogs
    **{g: "Phyllomedusidae" for g in [
        "Phyllomedusa", "Agalychnis", "Pithecopus", "Callimedusa",
        "Cruziohyla",
    ]},

    # Rhinophrynidae — Mexican Burrowing Toad
    **{g: "Rhinophrynidae" for g in [
        "Rhinophrynus",
    ]},

    # ===== CAUDATA (Salamanders & Newts) =====

    # Ambystomatidae — Mole Salamanders
    **{g: "Ambystomatidae" for g in [
        "Ambystoma",
    ]},

    # Salamandridae — Newts & Fire Salamanders
    **{g: "Salamandridae" for g in [
        "Notophthalmus", "Taricha", "Triturus", "Lissotriton",
        "Ichthyosaura", "Ommatotriton", "Salamandra", "Salamandrina",
        "Cynops", "Paramesotriton", "Tylototriton", "Neurergus",
        "Calotriton", "Chioglossa", "Euproctus", "Pleurodeles",
        "Lyciasalamandra",
    ]},

    # Plethodontidae — Lungless Salamanders
    **{g: "Plethodontidae" for g in [
        "Plethodon", "Desmognathus", "Eurycea", "Pseudotriton",
        "Gyrinophilus", "Hemidactylium", "Aneides", "Batrachoseps",
        "Ensatina", "Hydromantes", "Speleomantes", "Bolitoglossa",
        "Chiropterotriton", "Nototriton", "Oedipina", "Pseudoeurycea",
        "Thorius", "Stereochilus", "Urspelerpes",
    ]},

    # Cryptobranchidae — Giant Salamanders
    **{g: "Cryptobranchidae" for g in [
        "Cryptobranchus", "Andrias",
    ]},

    # Proteidae — Mudpuppies & Olm
    **{g: "Proteidae" for g in [
        "Necturus", "Proteus",
    ]},

    # Sirenidae — Sirens
    **{g: "Sirenidae" for g in [
        "Siren", "Pseudobranchus",
    ]},

    # Amphiumidae — Amphiumas (Congo Eels)
    **{g: "Amphiumidae" for g in [
        "Amphiuma",
    ]},

    # Rhyacotritonidae — Torrent Salamanders
    **{g: "Rhyacotritonidae" for g in [
        "Rhyacotriton",
    ]},

    # Dicamptodontidae — Pacific Giant Salamanders
    **{g: "Dicamptodontidae" for g in [
        "Dicamptodon",
    ]},

    # Hynobiidae — Asiatic Salamanders
    **{g: "Hynobiidae" for g in [
        "Hynobius", "Onychodactylus", "Batrachuperus",
    ]},
}


# =============================================================================
# FAMILY → COMMON NAME  (what users will see)
# =============================================================================

FAMILY_COMMON = {
    # Frogs & Toads
    "Ranidae":               "True Frogs (Bullfrogs, Leopard Frogs)",
    "Hylidae":               "Tree Frogs (Spring Peepers, Gray Treefrogs)",
    "Bufonidae":             "True Toads (American Toad, Cane Toad)",
    "Eleutherodactylidae":   "Rain Frogs & Coquís",
    "Craugastoridae":        "Robber Frogs",
    "Microhylidae":          "Narrow-mouthed Frogs",
    "Leptodactylidae":       "Thin-toed Frogs & Whistling Frogs",
    "Dendrobatidae":         "Poison Dart Frogs",
    "Scaphiopodidae":        "Spadefoot Toads",
    "Pipidae":               "Tongueless Frogs (African Clawed Frog)",
    "Bombinatoridae":        "Fire-bellied Toads",
    "Alytidae":              "Midwife Toads & Painted Frogs",
    "Centrolenidae":         "Glass Frogs",
    "Rhacophoridae":         "Shrub Frogs & Flying Frogs",
    "Myobatrachidae":        "Australian Ground Frogs",
    "Pelodryadidae":         "Australasian Tree Frogs",
    "Limnodynastidae":       "Australian Swamp Frogs",
    "Pelodytidae":           "Parsley Frogs",
    "Pelobatidae":           "European Spadefoot Toads",
    "Arthroleptidae":        "Squeaker Frogs",
    "Hyperoliidae":          "African Reed Frogs",
    "Mantellidae":           "Malagasy Poison Frogs",
    "Hemisotidae":           "Shovelnose Frogs",
    "Ceratophryidae":        "Horned Frogs (Pacman Frogs)",
    "Dicroglossidae":        "Fork-tongued Frogs",
    "Pyxicephalidae":        "African Bullfrogs",
    "Phyllomedusidae":       "Leaf Frogs (Red-eyed Tree Frog)",
    "Rhinophrynidae":        "Mexican Burrowing Toad",

    # Salamanders & Newts
    "Ambystomatidae":        "Mole Salamanders (Tiger Salamander, Axolotl)",
    "Salamandridae":         "Newts & Fire Salamanders",
    "Plethodontidae":        "Lungless Salamanders",
    "Cryptobranchidae":      "Giant Salamanders (Hellbender)",
    "Proteidae":             "Mudpuppies & Waterdogs",
    "Sirenidae":             "Sirens",
    "Amphiumidae":           "Amphiumas (Congo Eels)",
    "Rhyacotritonidae":      "Torrent Salamanders",
    "Dicamptodontidae":      "Pacific Giant Salamanders",
    "Hynobiidae":            "Asiatic Salamanders",
}


def add_family_labels(csv_path, output_path=None):
    """Read merged_df, add family + family_common columns, save."""
    df = csv_path
    print(f"Loaded {len(df)} rows.")

    # Extract genus (first word of scientific_name)
    df["genus"] = df["scientific_name"].astype(str).str.split().str[0]

    # Map genus → family
    df["family"] = df["genus"].map(GENUS_TO_FAMILY)

    # Map family → common name
    df["family_common"] = df["family"].map(FAMILY_COMMON)

    # Report coverage
    mapped = df["family"].notna().sum()
    unmapped = df["family"].isna().sum()
    print(f"\nFamily mapping: {mapped} mapped, {unmapped} unmapped ({unmapped/len(df)*100:.1f}%)")

    if unmapped > 0:
        unknown_genera = df.loc[df["family"].isna(), "genus"].value_counts()
        print(f"\nUnmapped genera ({len(unknown_genera)} unique):")
        print(unknown_genera.to_string())

    # Distribution
    print(f"\n{'='*60}")
    print("FAMILY DISTRIBUTION (this is what your classifier will learn)")
    print(f"{'='*60}")
    family_counts = df["family"].value_counts()
    for fam, count in family_counts.items():
        common = FAMILY_COMMON.get(fam, "?")
        print(f"  {fam:25s} {count:5d}  — {common}")

    print(f"\nTotal families: {df['family'].nunique()}")
    print(f"Families with ≥30 clips: {(family_counts >= 30).sum()}")
    print(f"Families with ≥50 clips: {(family_counts >= 50).sum()}")

    # Also show common_name distribution within top families
    print(f"\n{'='*60}")
    print("SPECIES WITHIN TOP 5 FAMILIES")
    print(f"{'='*60}")
    for fam in family_counts.head(5).index:
        print(f"\n  {fam} ({FAMILY_COMMON.get(fam, '?')}):")
        species = df[df["family"] == fam]["common_name"].value_counts()
        for sp, ct in species.head(10).items():
            print(f"    {sp:40s} {ct:4d}")

    # Save
    if output_path is None:
        output_path = csv_path.replace(".csv", "_with_family.csv")
    df.to_csv(output_path)
    print(f"\nSaved to {output_path}")

    return df


if __name__ == "__main__":
    add_family_labels(merged_df)


Loaded 4699 rows.

Family mapping: 4699 mapped, 0 unmapped (0.0%)

FAMILY DISTRIBUTION (this is what your classifier will learn)
  Hylidae                    2817  — Tree Frogs (Spring Peepers, Gray Treefrogs)
  Ranidae                     997  — True Frogs (Bullfrogs, Leopard Frogs)
  Bufonidae                   517  — True Toads (American Toad, Cane Toad)
  Microhylidae                199  — Narrow-mouthed Frogs
  Eleutherodactylidae         137  — Rain Frogs & Coquís
  Scaphiopodidae               27  — Spadefoot Toads
  Craugastoridae                2  — Robber Frogs
  Sirenidae                     1  — Sirens
  Rhinophrynidae                1  — Mexican Burrowing Toad
  Leptodactylidae               1  — Thin-toed Frogs & Whistling Frogs

Total families: 10
Families with ≥30 clips: 5
Families with ≥50 clips: 5

SPECIES WITHIN TOP 5 FAMILIES

  Hylidae (Tree Frogs (Spring Peepers, Gray Treefrogs)):
    Spring Peeper                            1033
    Pacific chorus frog           

TypeError: argument of type 'method' is not iterable

In [153]:
merged_df.to_csv("merged_df_with_family.csv")
print("Saved!")

Saved!


In [154]:
merged_df.to_csv("merged_df.csv")
print(f"Saved {len(merged_df)} rows")

Saved 4699 rows
